# Brain Tumor MRI Segmentation — EfficientNetB0 + U-Net (Recall-Aware)

**Fixed architecture:** EfficientNetB0 encoder (ImageNet) + lightweight U-Net decoder.
**Research question:** Can a false-negative-aware loss (Tversky / Focal Tversky), optionally combined
with boundary supervision, reduce tumor under-segmentation (improve recall) relative to the existing
Dice-heavy U-Net baseline, while preserving acceptable precision?

**Existing baseline (reused, not retrained):** Precision = 0.91, Recall = 0.86

**Experiment budget (Colab-friendly):**
1. EfficientNetB0 + U-Net + BCE+Dice
2. EfficientNetB0 + U-Net + Tversky (β > α, recall-oriented)
3. Final model = best-performing loss from (1)/(2), + boundary loss **only if it helps**

Each run: staged transfer learning (frozen encoder → partial fine-tune), mixed precision,
early stopping, `ReduceLROnPlateau`, 256×256 inputs, `tf.data` pipeline with caching/prefetch.

Threshold selection is done **once**, post-hoc, on validation probability maps — no retraining.


## 0. Setup

In [ ]:
!pip install -q tensorflow-addons 2>/dev/null || true

import os, json, random, math, time
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, mixed_precision

print("TF version:", tf.__version__)
gpus = tf.config.list_physical_devices('GPU')
print("GPUs:", gpus)

SEED = 42
random.seed(SEED); np.random.seed(SEED); tf.random.set_seed(SEED)

# Mixed precision if a GPU is available (skip on CPU-only runtimes)
if gpus:
    mixed_precision.set_global_policy("mixed_float16")
    print("Mixed precision enabled:", mixed_precision.global_policy())
else:
    print("No GPU found — running in float32. Consider Runtime > Change runtime type > GPU.")


## 1. Config

Edit `DATA_DIR` to point at your dataset. Expected layout (adjust the loader below if yours differs):

```
DATA_DIR/
  images/   *.png or *.jpg   (MRI slices)
  masks/    *.png            (binary tumor masks, same filename stem as the matching image)
```


In [ ]:
from dataclasses import dataclass

@dataclass
class CFG:
    DATA_DIR: str = "/content/drive/MyDrive/brain_tumor_dataset"  # <-- EDIT ME
    IMAGES_SUBDIR: str = "images"
    MASKS_SUBDIR: str = "masks"
    IMG_SIZE: tuple = (256, 256)
    CHANNELS: int = 3
    BATCH_SIZE: int = 16          # drop to 8 if you hit OOM
    VAL_SPLIT: float = 0.15
    TEST_SPLIT: float = 0.15
    AUTOTUNE = tf.data.AUTOTUNE
    EPOCHS_STAGE1: int = 8        # encoder frozen
    EPOCHS_STAGE2: int = 8        # top encoder blocks unfrozen, low LR
    EARLY_STOP_PATIENCE: int = 4
    LR_PLATEAU_PATIENCE: int = 2
    STAGE1_LR: float = 1e-3
    STAGE2_LR: float = 1e-5
    OUTPUT_DIR: str = "/content/outputs"
    EXISTING_BASELINE = {"precision": 0.91, "recall": 0.86}  # reused, not retrained

cfg = CFG()
os.makedirs(cfg.OUTPUT_DIR, exist_ok=True)


## 2. Data pipeline (`tf.data`)

Lightweight, cached, prefetched. Augmentation is deliberately small: rotation, horizontal flip,
zoom, brightness/contrast — nothing expensive.

Adjust `_list_pairs` if your image/mask filenames don't share a stem, or if masks are multi-class
(this pipeline assumes a binary tumor mask).


In [ ]:
def _list_pairs(cfg):
    img_dir = os.path.join(cfg.DATA_DIR, cfg.IMAGES_SUBDIR)
    mask_dir = os.path.join(cfg.DATA_DIR, cfg.MASKS_SUBDIR)
    img_files = sorted([f for f in os.listdir(img_dir) if f.lower().endswith((".png", ".jpg", ".jpeg"))])
    pairs = []
    for f in img_files:
        stem = os.path.splitext(f)[0]
        # try same extension first, then common alternatives
        candidates = [stem + ext for ext in (".png", ".jpg", ".jpeg")]
        match = next((c for c in candidates if os.path.exists(os.path.join(mask_dir, c))), None)
        if match:
            pairs.append((os.path.join(img_dir, f), os.path.join(mask_dir, match)))
    if not pairs:
        raise FileNotFoundError(
            f"No image/mask pairs found under {cfg.DATA_DIR}. "
            f"Check IMAGES_SUBDIR/MASKS_SUBDIR and filename matching."
        )
    return pairs


def _split_pairs(pairs, cfg, seed=SEED):
    rng = random.Random(seed)
    pairs = pairs[:]
    rng.shuffle(pairs)
    n = len(pairs)
    n_val = int(n * cfg.VAL_SPLIT)
    n_test = int(n * cfg.TEST_SPLIT)
    test_pairs = pairs[:n_test]
    val_pairs = pairs[n_test:n_test + n_val]
    train_pairs = pairs[n_test + n_val:]
    return train_pairs, val_pairs, test_pairs


def _decode_image(path, channels, size):
    img = tf.io.read_file(path)
    img = tf.io.decode_image(img, channels=channels, expand_animations=False)
    img = tf.image.resize(img, size, method="bilinear")
    img = tf.cast(img, tf.float32) / 255.0
    img.set_shape([size[0], size[1], channels])
    return img


def _decode_mask(path, size):
    m = tf.io.read_file(path)
    m = tf.io.decode_image(m, channels=1, expand_animations=False)
    m = tf.image.resize(m, size, method="nearest")
    m = tf.cast(m > 127, tf.float32)
    m.set_shape([size[0], size[1], 1])
    return m


def _load_pair(img_path, mask_path, cfg):
    img = _decode_image(img_path, cfg.CHANNELS, cfg.IMG_SIZE)
    mask = _decode_mask(mask_path, cfg.IMG_SIZE)
    return img, mask


def _augment(img, mask):
    # small, medically reasonable augmentations only
    if tf.random.uniform([]) > 0.5:
        img = tf.image.flip_left_right(img)
        mask = tf.image.flip_left_right(mask)

    k = tf.random.uniform([], 0, 4, dtype=tf.int32)  # 0/90/180/270-ish via small-angle proxy skipped;
    # use small-angle rotation instead of full 90-degree steps for anatomical plausibility
    angle = tf.random.uniform([], -0.1, 0.1)  # radians, ~ +/-5.7 degrees
    img = tf.image.stateless_random_brightness(img, max_delta=0.08, seed=[k, 1])
    img = tf.image.stateless_random_contrast(img, lower=0.9, upper=1.1, seed=[k, 2])
    img = tf.clip_by_value(img, 0.0, 1.0)

    # small zoom via random crop + resize back
    if tf.random.uniform([]) > 0.5:
        size = cfg.IMG_SIZE[0]
        crop_frac = tf.random.uniform([], 0.9, 1.0)
        crop_size = tf.cast(tf.cast(size, tf.float32) * crop_frac, tf.int32)
        stacked = tf.concat([img, mask], axis=-1)
        stacked = tf.image.random_crop(stacked, size=[crop_size, crop_size, cfg.CHANNELS + 1])
        stacked = tf.image.resize(stacked, cfg.IMG_SIZE, method="nearest")
        img, mask = stacked[..., :cfg.CHANNELS], stacked[..., cfg.CHANNELS:]
        mask = tf.cast(mask > 0.5, tf.float32)

    return img, mask


def make_dataset(pairs, cfg, training=False):
    img_paths = [p[0] for p in pairs]
    mask_paths = [p[1] for p in pairs]
    ds = tf.data.Dataset.from_tensor_slices((img_paths, mask_paths))
    ds = ds.map(lambda i, m: _load_pair(i, m, cfg), num_parallel_calls=cfg.AUTOTUNE)
    if training:
        ds = ds.cache()
        ds = ds.shuffle(buffer_size=min(1000, len(pairs)), seed=SEED, reshuffle_each_iteration=True)
        ds = ds.map(_augment, num_parallel_calls=cfg.AUTOTUNE)
    else:
        ds = ds.cache()
    ds = ds.batch(cfg.BATCH_SIZE)
    ds = ds.prefetch(cfg.AUTOTUNE)
    return ds


# Build datasets
all_pairs = _list_pairs(cfg)
train_pairs, val_pairs, test_pairs = _split_pairs(all_pairs, cfg)
print(f"train={len(train_pairs)}  val={len(val_pairs)}  test={len(test_pairs)}")

train_ds = make_dataset(train_pairs, cfg, training=True)
val_ds = make_dataset(val_pairs, cfg, training=False)
test_ds = make_dataset(test_pairs, cfg, training=False)


## 3. Model — EfficientNetB0 encoder + U-Net decoder

Skip connections are pulled from four EfficientNetB0 activation blocks at decreasing spatial
resolution (matching the standard U-Net-style skip pattern). Decoder is a lightweight stack of
`Conv2DTranspose` upsampling blocks. An optional lightweight attention gate can be applied to the
skip connections; keep it off unless it's specifically being ablated (`USE_ATTENTION`).


In [ ]:
USE_ATTENTION = False  # set True only if explicitly ablating attention on skip connections

def attention_gate(x, g, inter_channels):
    """Lightweight additive attention gate (Oktay et al.-style), applied to a skip connection."""
    theta_x = layers.Conv2D(inter_channels, 1, padding="same")(x)
    phi_g = layers.Conv2D(inter_channels, 1, padding="same")(g)
    phi_g = layers.Resizing(x.shape[1], x.shape[2], interpolation="bilinear")(phi_g)
    add = layers.Activation("relu")(layers.Add()([theta_x, phi_g]))
    psi = layers.Conv2D(1, 1, padding="same", activation="sigmoid")(add)
    return layers.Multiply()([x, psi])


def conv_block(x, filters, name):
    x = layers.Conv2D(filters, 3, padding="same", use_bias=False, name=f"{name}_conv1")(x)
    x = layers.BatchNormalization(name=f"{name}_bn1")(x)
    x = layers.Activation("relu", name=f"{name}_act1")(x)
    x = layers.Conv2D(filters, 3, padding="same", use_bias=False, name=f"{name}_conv2")(x)
    x = layers.BatchNormalization(name=f"{name}_bn2")(x)
    x = layers.Activation("relu", name=f"{name}_act2")(x)
    return x


def decoder_block(x, skip, filters, name, use_attention=False):
    x = layers.Conv2DTranspose(filters, 2, strides=2, padding="same", name=f"{name}_upconv")(x)
    if skip is not None:
        if use_attention:
            skip = attention_gate(skip, x, max(filters // 2, 8))
        x = layers.Concatenate(name=f"{name}_concat")([x, skip])
    x = conv_block(x, filters, name)
    return x


def build_effnetb0_unet(input_shape=(256, 256, 3), use_attention=USE_ATTENTION):
    inputs = layers.Input(shape=input_shape, name="mri_input")

    backbone = keras.applications.EfficientNetB0(
        include_top=False, weights="imagenet", input_tensor=inputs
    )

    # Skip-connection feature maps at decreasing resolution (EfficientNetB0 block outputs)
    skip_names = [
        "block2a_expand_activation",  # ~128x128
        "block3a_expand_activation",  # ~64x64
        "block4a_expand_activation",  # ~32x32
        "block6a_expand_activation",  # ~16x16
    ]
    skips = [backbone.get_layer(n).output for n in skip_names]
    bottleneck = backbone.output  # ~8x8

    x = bottleneck
    filters = [256, 128, 64, 32]
    for f, skip in zip(filters, reversed(skips)):
        x = decoder_block(x, skip, f, name=f"dec{f}", use_attention=use_attention)

    # one more upsample stage to get back to full 256x256 resolution
    x = layers.Conv2DTranspose(16, 2, strides=2, padding="same", name="dec_final_upconv")(x)
    x = conv_block(x, 16, name="dec_final")

    outputs = layers.Conv2D(1, 1, padding="same", name="seg_logits")(x)
    outputs = layers.Activation("sigmoid", dtype="float32", name="seg_output")(outputs)

    model = keras.Model(inputs, outputs, name="EfficientNetB0_UNet")
    return model, backbone


model, backbone = build_effnetb0_unet(input_shape=(*cfg.IMG_SIZE, cfg.CHANNELS))
model.summary()


## 4. Losses

- `bce_dice_loss` — Experiment 1
- `tversky_loss(alpha=0.3, beta=0.7)` — Experiment 2 (recall-oriented: β > α penalizes false negatives more)
- `focal_tversky_loss` — used in the final model only if Tversky shows a real recall gain
- `boundary_loss` — added to the final model only if it helps precision/boundary quality; ablate, don't assume


In [ ]:
EPS = 1e-6

def dice_coef(y_true, y_pred, smooth=1.0):
    y_true_f = tf.reshape(y_true, [-1])
    y_pred_f = tf.reshape(y_pred, [-1])
    intersection = tf.reduce_sum(y_true_f * y_pred_f)
    return (2.0 * intersection + smooth) / (tf.reduce_sum(y_true_f) + tf.reduce_sum(y_pred_f) + smooth)


def dice_loss(y_true, y_pred):
    return 1.0 - dice_coef(y_true, y_pred)


def bce_dice_loss(y_true, y_pred):
    bce = keras.losses.binary_crossentropy(y_true, y_pred)
    bce = tf.reduce_mean(bce)
    return bce + dice_loss(y_true, y_pred)


def tversky_index(y_true, y_pred, alpha=0.3, beta=0.7, smooth=1.0):
    y_true_f = tf.reshape(y_true, [-1])
    y_pred_f = tf.reshape(y_pred, [-1])
    tp = tf.reduce_sum(y_true_f * y_pred_f)
    fp = tf.reduce_sum((1 - y_true_f) * y_pred_f)
    fn = tf.reduce_sum(y_true_f * (1 - y_pred_f))
    return (tp + smooth) / (tp + alpha * fp + beta * fn + smooth)


def tversky_loss(alpha=0.3, beta=0.7):
    def _loss(y_true, y_pred):
        return 1.0 - tversky_index(y_true, y_pred, alpha=alpha, beta=beta)
    return _loss


def focal_tversky_loss(alpha=0.3, beta=0.7, gamma=0.75):
    def _loss(y_true, y_pred):
        ti = tversky_index(y_true, y_pred, alpha=alpha, beta=beta)
        return tf.pow((1.0 - ti), gamma)
    return _loss


def boundary_loss_fn():
    """Approximate boundary-aware loss via Sobel-gradient difference between GT and prediction masks.
    Lightweight — no distance-transform precomputation required, keeping it Colab-friendly."""
    def _loss(y_true, y_pred):
        true_edges = tf.image.sobel_edges(y_true)
        pred_edges = tf.image.sobel_edges(y_pred)
        true_mag = tf.sqrt(tf.reduce_sum(tf.square(true_edges), axis=-1) + EPS)
        pred_mag = tf.sqrt(tf.reduce_sum(tf.square(pred_edges), axis=-1) + EPS)
        return tf.reduce_mean(tf.abs(true_mag - pred_mag))
    return _loss


def combined_loss(base_loss_fn, boundary_weight=0.0):
    b_loss = boundary_loss_fn()
    def _loss(y_true, y_pred):
        l = base_loss_fn(y_true, y_pred)
        if boundary_weight > 0.0:
            l = l + boundary_weight * b_loss(y_true, y_pred)
        return l
    return _loss


## 5. Metrics

In [ ]:
def _binarize(y_pred, threshold=0.5):
    return tf.cast(y_pred > threshold, tf.float32)

def precision_m(y_true, y_pred, threshold=0.5):
    y_pred_b = _binarize(y_pred, threshold)
    tp = tf.reduce_sum(y_true * y_pred_b)
    fp = tf.reduce_sum((1 - y_true) * y_pred_b)
    return (tp + EPS) / (tp + fp + EPS)

def recall_m(y_true, y_pred, threshold=0.5):
    y_pred_b = _binarize(y_pred, threshold)
    tp = tf.reduce_sum(y_true * y_pred_b)
    fn = tf.reduce_sum(y_true * (1 - y_pred_b))
    return (tp + EPS) / (tp + fn + EPS)

def specificity_m(y_true, y_pred, threshold=0.5):
    y_pred_b = _binarize(y_pred, threshold)
    tn = tf.reduce_sum((1 - y_true) * (1 - y_pred_b))
    fp = tf.reduce_sum((1 - y_true) * y_pred_b)
    return (tn + EPS) / (tn + fp + EPS)

def iou_m(y_true, y_pred, threshold=0.5):
    y_pred_b = _binarize(y_pred, threshold)
    intersection = tf.reduce_sum(y_true * y_pred_b)
    union = tf.reduce_sum(y_true) + tf.reduce_sum(y_pred_b) - intersection
    return (intersection + EPS) / (union + EPS)

def f1_m(y_true, y_pred, threshold=0.5):
    p = precision_m(y_true, y_pred, threshold)
    r = recall_m(y_true, y_pred, threshold)
    return (2 * p * r + EPS) / (p + r + EPS)

def evaluate_dataset(model, dataset, threshold=0.5):
    """Full-pass evaluation over a tf.data dataset, returns a metrics dict (single pass, batched)."""
    tp = fp = fn = tn = 0.0
    dice_num = dice_den = 0.0
    for imgs, masks in dataset:
        preds = model(imgs, training=False)
        preds_b = _binarize(preds, threshold)
        tp += float(tf.reduce_sum(masks * preds_b))
        fp += float(tf.reduce_sum((1 - masks) * preds_b))
        fn += float(tf.reduce_sum(masks * (1 - preds_b)))
        tn += float(tf.reduce_sum((1 - masks) * (1 - preds_b)))
        dice_num += float(2 * tf.reduce_sum(masks * preds_b))
        dice_den += float(tf.reduce_sum(masks) + tf.reduce_sum(preds_b))

    precision = tp / (tp + fp + EPS)
    recall = tp / (tp + fn + EPS)
    specificity = tn / (tn + fp + EPS)
    iou = tp / (tp + fp + fn + EPS)
    dice = dice_num / (dice_den + EPS)
    f1 = (2 * precision * recall) / (precision + recall + EPS)

    return {
        "threshold": threshold,
        "precision": precision,
        "recall": recall,
        "sensitivity": recall,  # sensitivity == recall for binary segmentation
        "specificity": specificity,
        "iou": iou,
        "dice": dice,
        "f1": f1,
    }


## 6. Staged training routine

Stage 1: encoder frozen, decoder trained.
Stage 2: top encoder blocks unfrozen, very low LR fine-tune.
Both stages use early stopping + `ReduceLROnPlateau`, capped epoch budget.


In [ ]:
def compile_model(model, loss_fn, lr):
    optimizer = keras.optimizers.Adam(learning_rate=lr)
    model.compile(
        optimizer=optimizer,
        loss=loss_fn,
        metrics=[dice_coef, precision_m, recall_m, iou_m],
    )
    return model


def train_experiment(exp_name, model, backbone, loss_fn, train_ds, val_ds, cfg,
                      unfreeze_from_block=None):
    """Runs the two-stage training schedule for one experiment and returns (model, history_stage1,
    history_stage2, ckpt_path)."""
    ckpt_dir = os.path.join(cfg.OUTPUT_DIR, exp_name)
    os.makedirs(ckpt_dir, exist_ok=True)
    ckpt_path = os.path.join(ckpt_dir, "best.weights.h5")

    # --- Stage 1: freeze encoder ---
    backbone.trainable = False
    compile_model(model, loss_fn, cfg.STAGE1_LR)

    callbacks_1 = [
        keras.callbacks.EarlyStopping(monitor="val_loss", patience=cfg.EARLY_STOP_PATIENCE,
                                       restore_best_weights=True),
        keras.callbacks.ReduceLROnPlateau(monitor="val_loss", patience=cfg.LR_PLATEAU_PATIENCE,
                                           factor=0.5, min_lr=1e-7),
        keras.callbacks.ModelCheckpoint(ckpt_path, monitor="val_loss", save_best_only=True,
                                         save_weights_only=True),
    ]
    print(f"[{exp_name}] Stage 1 — encoder frozen")
    hist1 = model.fit(train_ds, validation_data=val_ds, epochs=cfg.EPOCHS_STAGE1,
                       callbacks=callbacks_1, verbose=2)

    # --- Stage 2: unfreeze top encoder blocks, low LR ---
    backbone.trainable = True
    if unfreeze_from_block is not None:
        for layer in backbone.layers:
            if unfreeze_from_block not in layer.name:
                layer.trainable = False
            # once we hit the target block name, everything from here on stays trainable
        # simpler + robust alternative: freeze everything before the named block
        found = False
        for layer in backbone.layers:
            if unfreeze_from_block in layer.name:
                found = True
            layer.trainable = found
    compile_model(model, loss_fn, cfg.STAGE2_LR)

    callbacks_2 = [
        keras.callbacks.EarlyStopping(monitor="val_loss", patience=cfg.EARLY_STOP_PATIENCE,
                                       restore_best_weights=True),
        keras.callbacks.ReduceLROnPlateau(monitor="val_loss", patience=cfg.LR_PLATEAU_PATIENCE,
                                           factor=0.5, min_lr=1e-7),
        keras.callbacks.ModelCheckpoint(ckpt_path, monitor="val_loss", save_best_only=True,
                                         save_weights_only=True),
    ]
    print(f"[{exp_name}] Stage 2 — partial fine-tune")
    hist2 = model.fit(train_ds, validation_data=val_ds, epochs=cfg.EPOCHS_STAGE2,
                       callbacks=callbacks_2, verbose=2)

    model.load_weights(ckpt_path)
    return model, hist1, hist2, ckpt_path


## 7. Run experiments

Reuses the existing baseline metrics (no retraining). Trains Experiment 1 (BCE+Dice) and
Experiment 2 (Tversky, β>α). The final model is only assembled from whichever of the two
performs better on recall, and boundary loss is only kept if it demonstrably helps.


In [ ]:
results = {"existing_baseline_unet_dice": cfg.EXISTING_BASELINE}

# ---- Experiment 1: EfficientNetB0 + U-Net + BCE+Dice ----
model_e1, backbone_e1 = build_effnetb0_unet(input_shape=(*cfg.IMG_SIZE, cfg.CHANNELS))
model_e1, h1a, h1b, ckpt_e1 = train_experiment(
    "exp1_bce_dice", model_e1, backbone_e1, bce_dice_loss,
    train_ds, val_ds, cfg, unfreeze_from_block="block6a"
)
results["exp1_bce_dice"] = evaluate_dataset(model_e1, val_ds)
print("Experiment 1 (val):", results["exp1_bce_dice"])


In [ ]:
# ---- Experiment 2: EfficientNetB0 + U-Net + Tversky (alpha=0.3, beta=0.7) ----
ALPHA, BETA = 0.3, 0.7  # beta > alpha => penalize false negatives more heavily

model_e2, backbone_e2 = build_effnetb0_unet(input_shape=(*cfg.IMG_SIZE, cfg.CHANNELS))
model_e2, h2a, h2b, ckpt_e2 = train_experiment(
    "exp2_tversky", model_e2, backbone_e2, tversky_loss(alpha=ALPHA, beta=BETA),
    train_ds, val_ds, cfg, unfreeze_from_block="block6a"
)
results["exp2_tversky"] = evaluate_dataset(model_e2, val_ds)
print("Experiment 2 (val):", results["exp2_tversky"])


### Compare Experiment 1 vs Experiment 2

Pick the loss that most improves recall without an unacceptable precision drop relative to the
baseline (0.91 / 0.86). This choice should be made from the printed numbers below — don't assume
Tversky wins; the data decides.


In [ ]:
print(json.dumps(results, indent=2, default=float))

# Manually inspect the two rows above, then set which one moves forward as the base loss
# for the final model. Example decision rule (edit thresholds to taste):
def pick_best_loss(results):
    r1, r2 = results["exp1_bce_dice"], results["exp2_tversky"]
    # prefer whichever has higher recall, as long as precision doesn't fall too far below baseline
    baseline_p = results["existing_baseline_unet_dice"]["precision"]
    candidates = [("exp1_bce_dice", r1, bce_dice_loss), ("exp2_tversky", r2, tversky_loss(ALPHA, BETA))]
    candidates = [c for c in candidates if c[1]["precision"] >= baseline_p - 0.05]
    if not candidates:
        candidates = [("exp1_bce_dice", r1, bce_dice_loss), ("exp2_tversky", r2, tversky_loss(ALPHA, BETA))]
    best_name, best_metrics, best_loss_fn = max(candidates, key=lambda c: c[1]["recall"])
    return best_name, best_loss_fn

best_loss_name, best_loss_fn = pick_best_loss(results)
print("Selected base loss for final model:", best_loss_name)


## 8. Final model — best loss ± boundary supervision

Trains **two** short final variants (with and without boundary loss added on top of the selected
base loss) and keeps boundary supervision only if it improves Dice/precision without hurting
recall meaningfully. If it doesn't help, the plain base-loss model stays final — per the "do not
introduce components that don't earn their place" rule.


In [ ]:
# Optionally switch base loss to Focal Tversky if Experiment 2 showed a clear recall gain
# over Experiment 1 (edit this flag manually after inspecting the results table above).
USE_FOCAL_TVERSKY_FOR_FINAL = False
if USE_FOCAL_TVERSKY_FOR_FINAL:
    final_base_loss = focal_tversky_loss(alpha=ALPHA, beta=BETA, gamma=0.75)
    final_base_name = "focal_tversky"
else:
    final_base_loss = best_loss_fn
    final_base_name = best_loss_name

# --- Final variant A: base loss only ---
model_final_a, backbone_final_a = build_effnetb0_unet(input_shape=(*cfg.IMG_SIZE, cfg.CHANNELS))
model_final_a, hfa1, hfa2, ckpt_final_a = train_experiment(
    "final_no_boundary", model_final_a, backbone_final_a, final_base_loss,
    train_ds, val_ds, cfg, unfreeze_from_block="block6a"
)
results["final_no_boundary"] = evaluate_dataset(model_final_a, val_ds)
print("Final (no boundary, val):", results["final_no_boundary"])


In [ ]:
# --- Final variant B: base loss + boundary loss ---
BOUNDARY_WEIGHT = 0.3
final_loss_with_boundary = combined_loss(final_base_loss, boundary_weight=BOUNDARY_WEIGHT)

model_final_b, backbone_final_b = build_effnetb0_unet(input_shape=(*cfg.IMG_SIZE, cfg.CHANNELS))
model_final_b, hfb1, hfb2, ckpt_final_b = train_experiment(
    "final_with_boundary", model_final_b, backbone_final_b, final_loss_with_boundary,
    train_ds, val_ds, cfg, unfreeze_from_block="block6a"
)
results["final_with_boundary"] = evaluate_dataset(model_final_b, val_ds)
print("Final (with boundary, val):", results["final_with_boundary"])

# Decide whether boundary loss earns its place
no_b, with_b = results["final_no_boundary"], results["final_with_boundary"]
keep_boundary = (with_b["dice"] > no_b["dice"]) and (with_b["recall"] >= no_b["recall"] - 0.02)
print("Keep boundary loss in final model:", keep_boundary)

final_model = model_final_b if keep_boundary else model_final_a
final_model_name = "final_with_boundary" if keep_boundary else "final_no_boundary"
print("Selected final model:", final_model_name)


## 9. Threshold optimization (no retraining)

Generate probability maps once on the validation set, sweep thresholds, pick the best by Dice
(or recall-weighted criterion of your choice), then evaluate that single threshold once on test.


In [ ]:
THRESHOLDS = [0.3, 0.4, 0.5, 0.6, 0.7]

threshold_results = []
for t in THRESHOLDS:
    m = evaluate_dataset(final_model, val_ds, threshold=t)
    threshold_results.append(m)
    print(f"threshold={t:.1f}  precision={m['precision']:.3f}  recall={m['recall']:.3f}  "
          f"dice={m['dice']:.3f}  iou={m['iou']:.3f}")

# Select threshold that maximizes Dice on the validation set (change the key if you prefer
# to optimize recall subject to a precision floor instead)
best_threshold_entry = max(threshold_results, key=lambda r: r["dice"])
BEST_THRESHOLD = best_threshold_entry["threshold"]
print("Selected threshold (by val Dice):", BEST_THRESHOLD)


## 10. Final test-set evaluation (single pass, selected threshold only)

In [ ]:
final_test_metrics = evaluate_dataset(final_model, test_ds, threshold=BEST_THRESHOLD)
print("FINAL TEST METRICS:")
print(json.dumps(final_test_metrics, indent=2, default=float))

results["final_test"] = final_test_metrics
with open(os.path.join(cfg.OUTPUT_DIR, "results_summary.json"), "w") as f:
    json.dump(results, f, indent=2, default=float)
print("Saved results_summary.json")


## 10b. Optional: HD95

Only compute if `scipy` is available and it's not too slow for your dataset size — this metric
is not required for the core hypothesis test.


In [ ]:
def hd95(y_true, y_pred):
    """95th-percentile Hausdorff distance between two binary masks (2D). Requires scipy."""
    from scipy.ndimage import distance_transform_edt
    y_true = y_true.astype(bool)
    y_pred = y_pred.astype(bool)
    if y_true.sum() == 0 or y_pred.sum() == 0:
        return np.nan
    dt_true = distance_transform_edt(~y_true)
    dt_pred = distance_transform_edt(~y_pred)
    d1 = dt_pred[y_true]
    d2 = dt_true[y_pred]
    all_d = np.concatenate([d1, d2])
    return float(np.percentile(all_d, 95))

# Example: compute average HD95 over the test set (comment out if too slow for your dataset)
hd95_values = []
for imgs, masks in test_ds:
    preds = final_model(imgs, training=False)
    preds_b = (preds.numpy() > BEST_THRESHOLD).astype(np.uint8)
    masks_np = masks.numpy().astype(np.uint8)
    for i in range(masks_np.shape[0]):
        hd95_values.append(hd95(masks_np[i, ..., 0], preds_b[i, ..., 0]))

hd95_values = [v for v in hd95_values if not np.isnan(v)]
if hd95_values:
    print("Mean HD95 (test):", float(np.mean(hd95_values)))
else:
    print("HD95 could not be computed (empty masks or predictions).")


## 11. Visual error analysis

A small representative set (5–10 cases): MRI / ground truth / prediction / false positives /
false negatives, spanning small/medium/large tumors, a difficult boundary case, and an
under-segmentation case where possible.


In [ ]:
import matplotlib.pyplot as plt

def show_case(img, mask, pred, threshold=0.5, title=""):
    pred_b = (pred > threshold).astype(np.float32)
    fp = ((pred_b == 1) & (mask == 0)).astype(np.float32)
    fn = ((pred_b == 0) & (mask == 1)).astype(np.float32)

    fig, axes = plt.subplots(1, 5, figsize=(15, 3))
    panels = [img, mask, pred_b, fp, fn]
    titles = ["MRI", "Ground Truth", "Prediction", "False Positive", "False Negative"]
    cmaps = [None, "gray", "gray", "Reds", "Blues"]
    for ax, panel, t, cmap in zip(axes, panels, titles, cmaps):
        if panel.shape[-1] == 1:
            panel = panel[..., 0]
        ax.imshow(panel, cmap=cmap)
        ax.set_title(t, fontsize=9)
        ax.axis("off")
    fig.suptitle(title, fontsize=10)
    plt.tight_layout()
    plt.show()


def pick_representative_cases(dataset, model, threshold, n=8):
    """Grab a handful of cases spanning a range of tumor sizes / segmentation difficulty."""
    records = []
    for imgs, masks in dataset:
        preds = model(imgs, training=False).numpy()
        imgs_np = imgs.numpy()
        masks_np = masks.numpy()
        for i in range(imgs_np.shape[0]):
            tumor_area = masks_np[i].sum()
            pred_b = (preds[i] > threshold).astype(np.float32)
            fn_area = ((pred_b == 0) & (masks_np[i] == 1)).sum()
            records.append({
                "img": imgs_np[i], "mask": masks_np[i], "pred": preds[i],
                "tumor_area": tumor_area, "fn_area": fn_area,
            })
    records.sort(key=lambda r: r["tumor_area"])
    if not records:
        return []
    picks = []
    picks.append(records[len(records) // 8])         # small tumor
    picks.append(records[len(records) // 2])          # medium tumor
    picks.append(records[-max(1, len(records) // 8)]) # large tumor
    hardest = max(records, key=lambda r: r["fn_area"])
    picks.append(hardest)                               # worst under-segmentation
    remaining = [r for r in records if r not in picks]
    picks.extend(remaining[:max(0, n - len(picks))])
    return picks[:n]


cases = pick_representative_cases(test_ds, final_model, BEST_THRESHOLD, n=8)
for idx, c in enumerate(cases):
    show_case(c["img"], c["mask"], c["pred"], threshold=BEST_THRESHOLD,
              title=f"Case {idx+1} — tumor area={int(c['tumor_area'])}px, FN area={int(c['fn_area'])}px")


## 12. Summary

`results_summary.json` (in `cfg.OUTPUT_DIR`) contains: the reused existing baseline, Experiment 1,
Experiment 2, both final variants (with/without boundary loss), and the final test-set metrics at
the selected threshold. Report precision, recall, Dice, IoU, F1, sensitivity, specificity (and
HD95 if computed) for the final model, and state plainly whether the recall-aware hypothesis was
supported by the numbers — don't round up.
